# 03 — Trực quan hóa dữ liệu NUCLE

Chạy **Run All** với kernel Python của dự án (pandas, IPython và Matplotlib). Chạy `uv sync --locked` tại thư mục dự án để cài dependency. Biểu đồ được vẽ bằng Matplotlib. Mỗi annotation là một quan sát, không phải một câu độc lập. Notebook chỉ đọc dữ liệu, không xóa hoặc gộp annotation.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src/data/parse_nucle.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open the notebook from inside the project.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data.constants import SGML_FILE, TARGET_LABELS
from src.data.nucle_sgml import load_nucle
from src.data.parse_nucle import documents_to_dataframe

if not SGML_FILE.is_file():
    raise FileNotFoundError(f'Place the licensed NUCLE corpus at {SGML_FILE}')
docs = load_nucle(SGML_FILE)
if not docs:
    raise ValueError('No documents found in corpus.')
df = documents_to_dataframe(docs)
target = df.loc[df.error_type.isin(TARGET_LABELS)].copy()
display(pd.Series({
    'Total essays': len(docs),
    'Total annotations': len(df),
    'Error labels': df.error_type.nunique(),
    'Target annotations': len(target),
    'Target coverage (%)': round(100 * len(target) / len(df), 2) if len(df) else 0,
}, name='Value').to_frame())


## Hàm vẽ biểu đồ

Thanh ngang bắt đầu từ 0. Số cuối thanh là giá trị chính xác. Màu xanh biểu thị phân bố; màu cam biểu thị cờ cần xem xét.


In [ ]:
def bar_chart(values, title, unit='annotation', color='#2563eb'):
    values = pd.Series(values).fillna(0)
    if values.empty:
        print(f'{title}: no data.')
        return
    fig, ax = plt.subplots(figsize=(11, max(3, 0.35 * len(values) + 1.3)), layout='constrained')
    bars = ax.barh(range(len(values)), values.to_numpy(dtype=float), color=color)
    ax.set_yticks(range(len(values)), labels=[str(label) for label in values.index])
    ax.invert_yaxis()
    ax.bar_label(bars, labels=[f'{value:,.0f}' for value in values], padding=5)
    ax.set_xlim(0, max(float(values.max()), 1) * 1.18)
    ax.set(title=title, xlabel=f'Count ({unit})')
    ax.xaxis.grid(True, alpha=0.2)
    ax.set_axisbelow(True)
    plt.show()
    plt.close(fig)


## 1. Phân bố tất cả nhãn lỗi

Đếm toàn bộ annotation, bao gồm dòng có cờ chất lượng. Chênh lệch độ dài thanh thể hiện mất cân bằng nhãn.


In [ ]:
all_counts = df.error_type.value_counts(dropna=False)
bar_chart(all_counts, 'Annotations by error label')


## 2. Năm nhãn mục tiêu

Phần trăm nhóm mục tiêu và toàn corpus có mẫu số khác nhau. Số bài tính bằng essay_index để không gộp tài liệu trùng essay_id.


In [ ]:
counts = target.error_type.value_counts().reindex(TARGET_LABELS, fill_value=0)
summary = counts.rename('annotation').to_frame()
summary['% target subset'] = counts / max(len(target), 1) * 100
summary['% corpus'] = counts / max(len(df), 1) * 100
summary['Essays containing label'] = target.groupby('error_type').essay_index.nunique().reindex(TARGET_LABELS, fill_value=0)
bar_chart(counts, 'Five target labels')
display(summary.round(2))
positive = counts[counts > 0]
if len(positive) == len(TARGET_LABELS):
    print(f'Largest / smallest class ratio: {positive.max() / positive.min():.2f}')
else:
    print('Some target labels are absent; five-class ratio is undefined.')


## 3. Loại chỉnh sửa theo nhãn

Bảng màu hiển thị phần trăm theo hàng; hàng không có annotation giữ bằng 0. invalid_offset, cross_paragraph và missing_correction là trạng thái dữ liệu, không phải thao tác chỉnh sửa đã xác định.


In [ ]:
bar_chart(target.edit_type.value_counts(), 'Edit types - target annotations')
edit_table = pd.crosstab(target.error_type, target.edit_type).reindex(TARGET_LABELS, fill_value=0)
edit_pct = edit_table.div(edit_table.sum(axis=1).replace(0, float('nan')), axis=0).fillna(0) * 100
display(edit_table)
if edit_pct.shape[1] == 0:
    print('No target edits available for the heatmap.')
else:
    fig, ax = plt.subplots(figsize=(10, 5), layout='constrained')
    heatmap = ax.imshow(edit_pct.to_numpy(dtype=float), cmap='Blues', vmin=0, vmax=100, aspect='auto')
    ax.set_xticks(range(len(edit_pct.columns)), labels=edit_pct.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(edit_pct.index)), labels=edit_pct.index)
    for row in range(len(edit_pct.index)):
        for col in range(len(edit_pct.columns)):
            value = edit_pct.iloc[row, col]
            ax.text(col, row, f'{value:.1f}%', ha='center', va='center',
                    color='white' if value >= 60 else '#172033')
    ax.set(title='Edit type within each label (%)', xlabel='Edit type', ylabel='Error label')
    fig.colorbar(heatmap, ax=ax, label='Percentage within label (%)')
    plt.show()
    plt.close(fig)


## 4. Số annotation mục tiêu mỗi bài

Bao gồm tất cả bài trong SGML, kể cả bài có 0 lỗi mục tiêu. Các khoảng có độ rộng khác nhau: thanh biểu thị số bài, không phải mật độ xác suất.


In [ ]:
per_essay = target.groupby('essay_index').size().reindex(range(len(docs)), fill_value=0)
error_bins = [-1, 0, 5, 10, 20, 40, 80, float('inf')]
error_labels = ['0', '1-5', '6-10', '11-20', '21-40', '41-80', '>80']
grouped = pd.cut(per_essay, bins=error_bins, labels=error_labels).value_counts(sort=False)
bar_chart(grouped, 'Target annotations per essay', unit='essays')
display(per_essay.describe(percentiles=[.25, .5, .75, .95]).rename('Annotation / essays').to_frame())


## 5. Độ dài đoạn lỗi theo ký tự

Chỉ đo span có offset hợp lệ, nằm trong một đoạn và tái dựng được. Span rỗng hợp lệ dài 0, không thay span thiếu bằng chuỗi rỗng. Không cắt đuôi phân bố.


In [ ]:
valid_spans = target.loc[target.valid_offset & ~target.cross_paragraph & target.error_span.notna()].copy()
valid_spans['span_length'] = valid_spans.error_span.str.len()
print(f'Measured: {len(valid_spans):,}; Excluded: {len(target) - len(valid_spans):,}')
length_bins = [-1, 0, 5, 10, 20, 50, 100, float('inf')]
length_labels = ['0 (empty span)', '1-5', '6-10', '11-20', '21-50', '51-100', '>100']
length_counts = pd.cut(valid_spans.span_length, bins=length_bins, labels=length_labels).value_counts(sort=False)
bar_chart(length_counts, 'Error span length (characters)')
display(valid_spans.groupby('error_type').span_length.agg(['count', 'mean', 'median', 'max']).reindex(TARGET_LABELS).round(2))


## 6. Các cờ chất lượng trên toàn corpus

Một annotation có thể mang nhiều cờ, nên các nhóm không loại trừ nhau. Correction rỗng có thể là thao tác xóa hợp lệ, khác với thiếu thẻ correction. Cờ kỹ thuật không khẳng định lỗi ngữ nghĩa. Xem notebook 02_nucle_data_quality.ipynb để phân tích overlap và nghi trùng.


In [ ]:
flags = pd.Series({
    'Invalid offsets': int((~df.valid_offset).sum()),
    'Cross-paragraph annotations': int(df.cross_paragraph.sum()),
    'Missing correction tag': int(df.missing_correction.sum()),
    'Unreconstructed span': int(df.error_span.isna().sum()),
})
bar_chart(flags, 'Review flags - all annotations', color='#d97706')
display(pd.DataFrame({'Annotations': flags, '% corpus': flags / max(len(df), 1) * 100}).round(2))


## Gợi ý diễn giải

- Dùng phân bố nhãn để đánh giá mất cân bằng, không suy ra hiệu quả mô hình.
- So sánh số bài chứa nhãn với số annotation để xem mức tập trung lỗi.
- Kiểm tra span dài và cờ chất lượng trước khi quyết định tiền xử lý.
- Notebook không chia tập dữ liệu và không tự loại annotation.
